# MongoDB Estate Schema Inventory Notebook

This notebook inventories **every visible database and collection** on a MongoDB deployment and produces:

- database inventory
- collection/view inventory
- collection options, validators, and view/time-series metadata
- collection stats where permitted
- index catalog
- **observed schema** (field paths and BSON types discovered in documents)
- sample documents for each collection
- CSV, JSON, and Markdown exports in `mongo_schema_inventory_output/`

## Important note about "schema" in MongoDB

MongoDB is inherently flexible unless schema validators are enforced. For each collection, this notebook reports:

1. **Declared structure controls** such as validators, collection options, view definitions, and time-series options.
2. **Observed schema** inferred from stored documents.
3. **Sample documents** so you can review actual data shapes.

For strictest field discovery, set `SCAN_POLICY = "full"` in the config cell.  
For large estates, `SCAN_POLICY = "adaptive"` is safer and will fully scan smaller collections while sampling larger ones.

## If needed, install dependencies

Run this in a notebook cell if your environment is missing packages:

```python
%pip install pymongo pandas
```

In [1]:
import os
from pathlib import Path

# Connection
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")

# Output
OUTPUT_DIR = Path("mongo_schema_inventory_output")

# Database selection
INCLUDE_DATABASES = []     # Example: ["app_db", "analytics_db"] ; empty = all visible databases
EXCLUDE_DATABASES = []     # Example: ["scratch_db"]
SKIP_SYSTEM_DATABASES = False  # True to skip admin/config/local

# Scan behavior for schema inference
SCAN_POLICY = "adaptive"   # "full", "sample", or "adaptive"
ADAPTIVE_FULL_SCAN_MAX_DOCS = 25000
SAMPLE_SCAN_SIZE = 5000

# Sample documents written per collection
MAX_SAMPLE_DOCS_PER_COLLECTION = 5
SAMPLE_DOCUMENT_STRATEGY = "first"   # "first" or "random"

# Field reporting
FIELD_EXAMPLE_LIMIT = 3
TRUNCATE_LONG_VALUES_AT = 240

# Client timeouts
SERVER_SELECTION_TIMEOUT_MS = 15000
SOCKET_TIMEOUT_MS = 120000

# Set to True if you want a concise console log while scanning
VERBOSE = True

In [2]:
import json
import math
import re
from collections import defaultdict
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Markdown, display
from pymongo import MongoClient
from pymongo.errors import PyMongoError
from bson import json_util

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)


def log(message):
    if VERBOSE:
        print(message)


def safe_file_component(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("._")
    return cleaned or "unnamed"


def bson_type_name(value):
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, int) and not isinstance(value, bool):
        return "int"
    if isinstance(value, float):
        return "double"
    if isinstance(value, str):
        return "string"
    if isinstance(value, dict):
        return "object"
    if isinstance(value, list):
        return "array"
    if isinstance(value, bytes):
        return "bytes"
    class_name = value.__class__.__name__
    if class_name == "datetime":
        return "date"
    mapping = {
        "ObjectId": "objectId",
        "Int64": "long",
        "Decimal128": "decimal128",
        "Binary": "binary",
        "DBRef": "dbRef",
        "Regex": "regex",
        "Timestamp": "timestamp",
        "Code": "javascript",
        "MinKey": "minKey",
        "MaxKey": "maxKey",
    }
    return mapping.get(class_name, class_name)


def to_jsonable(value):
    try:
        return json.loads(json_util.dumps(value))
    except Exception:
        try:
            return json.loads(json.dumps(value, default=str))
        except Exception:
            return str(value)


def compact_json(value, limit=TRUNCATE_LONG_VALUES_AT):
    try:
        rendered = json.dumps(to_jsonable(value), ensure_ascii=False, default=str)
    except Exception:
        rendered = repr(value)
    rendered = re.sub(r"\s+", " ", rendered).strip()
    if len(rendered) > limit:
        return rendered[: max(0, limit - 3)] + "..."
    return rendered


def pretty_json_text(value):
    try:
        return json.dumps(to_jsonable(value), indent=2, ensure_ascii=False, default=str)
    except Exception:
        return repr(value)


def is_missing_scalar(value):
    if value is None:
        return True
    try:
        if isinstance(value, float) and math.isnan(value):
            return True
    except Exception:
        pass
    return False


def dataframe_to_markdown(df, max_rows=50):
    if df is None or df.empty:
        return "_No rows._"
    working = df.head(max_rows).copy()
    headers = [str(col) for col in working.columns]
    table_lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in working.itertuples(index=False):
        cells = []
        for value in row:
            text = "" if is_missing_scalar(value) else str(value)
            text = text.replace("\n", "<br>").replace("|", "\\|")
            cells.append(text)
        table_lines.append("| " + " | ".join(cells) + " |")
    if len(df) > max_rows:
        table_lines.append("")
        table_lines.append(f"_Showing first {max_rows} of {len(df)} rows._")
    return "\n".join(table_lines)


def extract_paths(value, path=""):
    if path:
        yield path, value
    if isinstance(value, dict):
        for key, child in value.items():
            child_path = f"{path}.{key}" if path else str(key)
            yield from extract_paths(child, child_path)
    elif isinstance(value, list):
        item_path = f"{path}[]" if path else "[]"
        for item in value:
            yield from extract_paths(item, item_path)


def new_field_entry():
    return {
        "documents_with_field": 0,
        "occurrences": 0,
        "non_null_occurrences": 0,
        "types": set(),
        "examples": [],
    }


def observe_document(doc, field_observations):
    seen_paths = set()
    for path, value in extract_paths(doc):
        entry = field_observations[path]
        if path not in seen_paths:
            entry["documents_with_field"] += 1
            seen_paths.add(path)
        entry["occurrences"] += 1
        if value is not None:
            entry["non_null_occurrences"] += 1
        entry["types"].add(bson_type_name(value))
        sample_value = compact_json(value)
        if sample_value and sample_value not in entry["examples"] and len(entry["examples"]) < FIELD_EXAMPLE_LIMIT:
            entry["examples"].append(sample_value)


def path_depth(path: str) -> int:
    return path.count(".") + path.count("[]")


def parent_path(path: str):
    if path.endswith("[]"):
        return path[:-2] or None
    idx = path.rfind(".")
    return path[:idx] if idx >= 0 else None


def safe_db_command(db, command_name, *args, **kwargs):
    try:
        if args:
            return db.command(command_name, *args, **kwargs), None
        return db.command(command_name, **kwargs), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def choose_scan_mode(estimated_count):
    policy = str(SCAN_POLICY).strip().lower()
    if policy == "full":
        return "full", None
    if policy == "sample":
        return "sample", int(SAMPLE_SCAN_SIZE)
    if estimated_count is not None and estimated_count <= int(ADAPTIVE_FULL_SCAN_MAX_DOCS):
        return "full", None
    return "sample", int(SAMPLE_SCAN_SIZE)


def get_visible_database_rows(client):
    rows = []
    try:
        result = client.admin.command("listDatabases", nameOnly=False)
        for item in result.get("databases", []):
            rows.append(
                {
                    "database": item.get("name"),
                    "sizeOnDisk": item.get("sizeOnDisk"),
                    "empty": item.get("empty"),
                    "list_source": "listDatabases",
                    "list_error": None,
                }
            )
    except Exception as exc:
        for name in client.list_database_names():
            rows.append(
                {
                    "database": name,
                    "sizeOnDisk": None,
                    "empty": None,
                    "list_source": "list_database_names",
                    "list_error": f"{type(exc).__name__}: {exc}",
                }
            )

    include = set(INCLUDE_DATABASES or [])
    exclude = set(EXCLUDE_DATABASES or [])
    system_dbs = {"admin", "config", "local"}

    filtered = []
    for row in rows:
        name = row["database"]
        if include and name not in include:
            continue
        if name in exclude:
            continue
        if SKIP_SYSTEM_DATABASES and name in system_dbs:
            continue
        filtered.append(row)

    return sorted(filtered, key=lambda r: r["database"])


def get_db_stats_row(db):
    stats, error = safe_db_command(db, "dbStats")
    if stats is None:
        return {
            "db_collections": None,
            "db_views": None,
            "db_objects": None,
            "db_avgObjSize": None,
            "db_dataSize": None,
            "db_storageSize": None,
            "db_indexes": None,
            "db_indexSize": None,
            "db_fsUsedSize": None,
            "db_fsTotalSize": None,
            "dbStats_error": error,
        }
    return {
        "db_collections": stats.get("collections"),
        "db_views": stats.get("views"),
        "db_objects": stats.get("objects"),
        "db_avgObjSize": stats.get("avgObjSize"),
        "db_dataSize": stats.get("dataSize"),
        "db_storageSize": stats.get("storageSize"),
        "db_indexes": stats.get("indexes"),
        "db_indexSize": stats.get("indexSize"),
        "db_fsUsedSize": stats.get("fsUsedSize"),
        "db_fsTotalSize": stats.get("fsTotalSize"),
        "dbStats_error": None,
    }


def get_collection_catalog_rows(db):
    rows = []
    try:
        for meta in db.list_collections():
            options = meta.get("options", {}) or {}
            info = meta.get("info", {}) or {}
            rows.append(
                {
                    "collection": meta.get("name"),
                    "collection_type": meta.get("type", "collection"),
                    "collection_options": options,
                    "validator": options.get("validator"),
                    "timeseries": options.get("timeseries"),
                    "view_on": options.get("viewOn"),
                    "view_pipeline": options.get("pipeline"),
                    "collation": options.get("collation"),
                    "change_stream_pre_and_post_images": options.get("changeStreamPreAndPostImages"),
                    "idIndex": meta.get("idIndex"),
                    "collection_info": info,
                    "catalog_error": None,
                }
            )
    except Exception as exc:
        return [], f"{type(exc).__name__}: {exc}"
    return rows, None


def get_collection_stats_row(db, collection_name):
    stats, error = safe_db_command(db, "collStats", collection_name)
    if stats is None:
        return {
            "coll_count": None,
            "coll_size": None,
            "coll_avgObjSize": None,
            "coll_storageSize": None,
            "coll_freeStorageSize": None,
            "coll_totalIndexSize": None,
            "coll_nindexes": None,
            "coll_capped": None,
            "coll_max": None,
            "coll_maxSize": None,
            "coll_timeseries_bucketNs": None,
            "coll_sharded": None,
            "collStats_error": error,
        }
    return {
        "coll_count": stats.get("count"),
        "coll_size": stats.get("size"),
        "coll_avgObjSize": stats.get("avgObjSize"),
        "coll_storageSize": stats.get("storageSize"),
        "coll_freeStorageSize": stats.get("freeStorageSize"),
        "coll_totalIndexSize": stats.get("totalIndexSize"),
        "coll_nindexes": stats.get("nindexes"),
        "coll_capped": stats.get("capped"),
        "coll_max": stats.get("max"),
        "coll_maxSize": stats.get("maxSize"),
        "coll_timeseries_bucketNs": ((stats.get("timeseries") or {}).get("bucketsNs") if isinstance(stats.get("timeseries"), dict) else None),
        "coll_sharded": stats.get("sharded"),
        "collStats_error": None,
    }


def get_estimated_document_count(collection, collection_type, coll_stats_row):
    count_from_stats = coll_stats_row.get("coll_count")
    if count_from_stats is not None:
        return count_from_stats
    if collection_type == "view":
        return None
    try:
        return collection.estimated_document_count()
    except Exception:
        return None


def get_sample_documents(collection, max_docs, strategy="first"):
    if not max_docs or int(max_docs) <= 0:
        return []
    max_docs = int(max_docs)
    strategy = str(strategy).strip().lower()
    if strategy == "random":
        try:
            return list(collection.aggregate([{"$sample": {"size": max_docs}}]))
        except Exception:
            pass
    return list(collection.find({}).limit(max_docs))


def save_json_file(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(pretty_json_text(payload), encoding="utf-8")


def save_sample_documents(db_name, collection_name, samples):
    path = OUTPUT_DIR / "samples" / safe_file_component(db_name) / f"{safe_file_component(collection_name)}.json"
    save_json_file(path, samples)
    return str(path)


def save_schema_observation(db_name, collection_name, collection_summary, field_observations):
    payload = {
        "database": db_name,
        "collection": collection_name,
        "scan_mode": collection_summary.get("scan_mode"),
        "scan_limit": collection_summary.get("scan_limit"),
        "documents_scanned_for_schema": collection_summary.get("documents_scanned_for_schema"),
        "estimated_document_count": collection_summary.get("estimated_document_count"),
        "schema_is_exhaustive_for_collection": collection_summary.get("schema_is_exhaustive_for_collection"),
        "fields": [],
    }
    for path in sorted(field_observations):
        entry = field_observations[path]
        payload["fields"].append(
            {
                "path": path,
                "parent_path": parent_path(path),
                "depth": path_depth(path),
                "documents_with_field": entry["documents_with_field"],
                "occurrences": entry["occurrences"],
                "non_null_occurrences": entry["non_null_occurrences"],
                "types": sorted(entry["types"]),
                "examples": entry["examples"],
            }
        )

    path = OUTPUT_DIR / "schemas" / safe_file_component(db_name) / f"{safe_file_component(collection_name)}.json"
    save_json_file(path, payload)
    return str(path)


def build_collection_row(database_name, catalog_row, coll_stats_row, estimated_count, scan_mode, scan_limit, scanned_docs, scan_error, sample_file, schema_file, index_error):
    schema_is_exhaustive = (
        scan_mode == "full"
        and scan_error is None
        and (
            estimated_count is None
            or scanned_docs == estimated_count
            or catalog_row.get("collection_type") == "view"
        )
    )

    return {
        "database": database_name,
        "collection": catalog_row.get("collection"),
        "collection_type": catalog_row.get("collection_type"),
        "estimated_document_count": estimated_count,
        "scan_mode": scan_mode,
        "scan_limit": scan_limit,
        "documents_scanned_for_schema": scanned_docs,
        "schema_is_exhaustive_for_collection": schema_is_exhaustive,
        "collection_options_json": compact_json(catalog_row.get("collection_options"), limit=5000),
        "validator_json": compact_json(catalog_row.get("validator"), limit=5000),
        "timeseries_json": compact_json(catalog_row.get("timeseries"), limit=5000),
        "view_on": catalog_row.get("view_on"),
        "view_pipeline_json": compact_json(catalog_row.get("view_pipeline"), limit=5000),
        "collation_json": compact_json(catalog_row.get("collation"), limit=5000),
        "idIndex_json": compact_json(catalog_row.get("idIndex"), limit=5000),
        "collection_info_json": compact_json(catalog_row.get("collection_info"), limit=5000),
        "sample_file": sample_file,
        "schema_file": schema_file,
        "scan_error": scan_error,
        "index_list_error": index_error,
        "catalog_error": catalog_row.get("catalog_error"),
        **coll_stats_row,
    }


def build_index_rows(database_name, collection_name, indexes):
    rows = []
    for idx in indexes:
        key_doc = idx.get("key", {})
        if hasattr(key_doc, "items"):
            key_text = ", ".join(f"{k}:{v}" for k, v in key_doc.items())
        else:
            key_text = compact_json(key_doc)
        rows.append(
            {
                "database": database_name,
                "collection": collection_name,
                "index_name": idx.get("name"),
                "index_keys": key_text,
                "unique": idx.get("unique", False),
                "sparse": idx.get("sparse", False),
                "hidden": idx.get("hidden", False),
                "expireAfterSeconds": idx.get("expireAfterSeconds"),
                "partialFilterExpression_json": compact_json(idx.get("partialFilterExpression"), limit=5000),
                "wildcardProjection_json": compact_json(idx.get("wildcardProjection"), limit=5000),
                "raw_index_json": compact_json(idx, limit=5000),
            }
        )
    return rows

In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=int(SERVER_SELECTION_TIMEOUT_MS),
    socketTimeoutMS=int(SOCKET_TIMEOUT_MS),
)

# Fail early if the connection is not available.
client.admin.command("ping")

run_started_utc = datetime.now(timezone.utc).isoformat()

database_rows = []
collection_rows = []
index_rows = []
field_rows = []

visible_databases = get_visible_database_rows(client)
log(f"Visible databases to inventory: {len(visible_databases)}")

for db_meta in visible_databases:
    db_name = db_meta["database"]
    db = client[db_name]
    db_row = {**db_meta, **get_db_stats_row(db)}
    database_rows.append(db_row)

    log("")
    log(f"=== Database: {db_name} ===")

    collection_catalog_rows, catalog_error = get_collection_catalog_rows(db)
    if catalog_error:
        log(f"  Could not list collections: {catalog_error}")
        collection_rows.append(
            {
                "database": db_name,
                "collection": None,
                "collection_type": None,
                "estimated_document_count": None,
                "scan_mode": None,
                "scan_limit": None,
                "documents_scanned_for_schema": 0,
                "schema_is_exhaustive_for_collection": False,
                "collection_options_json": None,
                "validator_json": None,
                "timeseries_json": None,
                "view_on": None,
                "view_pipeline_json": None,
                "collation_json": None,
                "idIndex_json": None,
                "collection_info_json": None,
                "sample_file": None,
                "schema_file": None,
                "scan_error": None,
                "index_list_error": None,
                "catalog_error": catalog_error,
                "coll_count": None,
                "coll_size": None,
                "coll_avgObjSize": None,
                "coll_storageSize": None,
                "coll_freeStorageSize": None,
                "coll_totalIndexSize": None,
                "coll_nindexes": None,
                "coll_capped": None,
                "coll_max": None,
                "coll_maxSize": None,
                "coll_timeseries_bucketNs": None,
                "coll_sharded": None,
                "collStats_error": None,
            }
        )
        continue

    if not collection_catalog_rows:
        log("  No collections or views found.")

    for catalog_row in collection_catalog_rows:
        collection_name = catalog_row["collection"]
        collection_type = catalog_row["collection_type"]
        coll = db[collection_name]

        coll_stats_row = get_collection_stats_row(db, collection_name)
        estimated_count = get_estimated_document_count(coll, collection_type, coll_stats_row)
        scan_mode, scan_limit = choose_scan_mode(estimated_count)

        log(f"  - {collection_name} ({collection_type}) | estimated docs={estimated_count} | scan={scan_mode}")

        # Sample documents
        sample_file = None
        try:
            samples = get_sample_documents(coll, MAX_SAMPLE_DOCS_PER_COLLECTION, SAMPLE_DOCUMENT_STRATEGY)
            sample_file = save_sample_documents(db_name, collection_name, samples)
        except Exception as exc:
            samples = []
            sample_file = None
            log(f"      sample error: {type(exc).__name__}: {exc}")

        # Indexes
        indexes = []
        index_error = None
        try:
            indexes = list(coll.list_indexes())
            index_rows.extend(build_index_rows(db_name, collection_name, indexes))
        except Exception as exc:
            index_error = f"{type(exc).__name__}: {exc}"
            log(f"      index error: {index_error}")

        # Schema scan
        field_observations = defaultdict(new_field_entry)
        scanned_docs = 0
        scan_error = None

        try:
            cursor = coll.find({})
            if scan_limit is not None:
                cursor = cursor.limit(int(scan_limit))
            for doc in cursor:
                scanned_docs += 1
                observe_document(doc, field_observations)
        except Exception as exc:
            scan_error = f"{type(exc).__name__}: {exc}"
            log(f"      scan error: {scan_error}")

        collection_summary = {
            "scan_mode": scan_mode,
            "scan_limit": scan_limit,
            "documents_scanned_for_schema": scanned_docs,
            "estimated_document_count": estimated_count,
            "schema_is_exhaustive_for_collection": (
                scan_mode == "full"
                and scan_error is None
                and (estimated_count is None or scanned_docs == estimated_count or collection_type == "view")
            ),
        }

        schema_file = save_schema_observation(db_name, collection_name, collection_summary, field_observations)

        collection_rows.append(
            build_collection_row(
                database_name=db_name,
                catalog_row=catalog_row,
                coll_stats_row=coll_stats_row,
                estimated_count=estimated_count,
                scan_mode=scan_mode,
                scan_limit=scan_limit,
                scanned_docs=scanned_docs,
                scan_error=scan_error,
                sample_file=sample_file,
                schema_file=schema_file,
                index_error=index_error,
            )
        )

        for path in sorted(field_observations):
            entry = field_observations[path]
            field_rows.append(
                {
                    "database": db_name,
                    "collection": collection_name,
                    "path": path,
                    "parent_path": parent_path(path),
                    "depth": path_depth(path),
                    "documents_scanned_for_schema": scanned_docs,
                    "documents_with_field": entry["documents_with_field"],
                    "presence_ratio_in_scan": (
                        round(entry["documents_with_field"] / scanned_docs, 6) if scanned_docs else None
                    ),
                    "observed_required_in_scan": bool(scanned_docs and entry["documents_with_field"] == scanned_docs),
                    "occurrences": entry["occurrences"],
                    "non_null_occurrences": entry["non_null_occurrences"],
                    "types": "|".join(sorted(entry["types"])),
                    "examples": " || ".join(entry["examples"]),
                    "scan_mode": scan_mode,
                    "estimated_document_count": estimated_count,
                }
            )

run_finished_utc = datetime.now(timezone.utc).isoformat()
log("")
log("Inventory complete.")

Visible databases to inventory: 4

=== Database: admin ===
  - system.version (collection) | estimated docs=2 | scan=full
  - system.users (collection) | estimated docs=1 | scan=full

=== Database: config ===
  - system.sessions (collection) | estimated docs=6 | scan=full
      sample error: OperationFailure: not authorized on config to execute command { find: "system.sessions", filter: {}, limit: 5, lsid: { id: UUID("bf7818fd-a205-41a1-b521-153a72212700") }, $db: "config" }, full error: {'ok': 0.0, 'errmsg': 'not authorized on config to execute command { find: "system.sessions", filter: {}, limit: 5, lsid: { id: UUID("bf7818fd-a205-41a1-b521-153a72212700") }, $db: "config" }', 'code': 13, 'codeName': 'Unauthorized'}
      scan error: OperationFailure: not authorized on config to execute command { find: "system.sessions", filter: {}, lsid: { id: UUID("bf7818fd-a205-41a1-b521-153a72212700") }, $db: "config" }, full error: {'ok': 0.0, 'errmsg': 'not authorized on config to execute comman

In [4]:
databases_df = pd.DataFrame(database_rows).sort_values(by=["database"], kind="stable").reset_index(drop=True)

collections_df = pd.DataFrame(collection_rows)
if not collections_df.empty:
    collections_df = collections_df.sort_values(by=["database", "collection"], kind="stable").reset_index(drop=True)

indexes_df = pd.DataFrame(index_rows)
if not indexes_df.empty:
    indexes_df = indexes_df.sort_values(by=["database", "collection", "index_name"], kind="stable").reset_index(drop=True)

fields_df = pd.DataFrame(field_rows)
if not fields_df.empty:
    fields_df = fields_df.sort_values(by=["database", "collection", "path"], kind="stable").reset_index(drop=True)

if not collections_df.empty and not fields_df.empty:
    field_count_df = (
        fields_df.groupby(["database", "collection"], dropna=False)
        .size()
        .reset_index(name="observed_field_paths")
    )
    collections_df = collections_df.merge(field_count_df, on=["database", "collection"], how="left")
else:
    if not collections_df.empty and "observed_field_paths" not in collections_df.columns:
        collections_df["observed_field_paths"] = None

summary_df = pd.DataFrame(
    [
        {
            "databases_visible": len(databases_df),
            "collections_and_views": int(collections_df["collection"].notna().sum()) if not collections_df.empty else 0,
            "indexes": int(indexes_df["index_name"].notna().sum()) if not indexes_df.empty else 0,
            "observed_field_paths": len(fields_df),
            "generated_utc": run_finished_utc,
        }
    ]
)

display(Markdown("# Inventory Summary"))
display(summary_df)

if not databases_df.empty:
    display(Markdown("## Databases"))
    display(
        databases_df[
            [
                "database",
                "sizeOnDisk",
                "empty",
                "db_collections",
                "db_views",
                "db_objects",
                "db_dataSize",
                "db_storageSize",
                "db_indexes",
                "db_indexSize",
                "dbStats_error",
            ]
        ]
    )

if not collections_df.empty:
    display(Markdown("## Collections / Views"))
    display(
        collections_df[
            [
                "database",
                "collection",
                "collection_type",
                "estimated_document_count",
                "scan_mode",
                "documents_scanned_for_schema",
                "schema_is_exhaustive_for_collection",
                "observed_field_paths",
                "coll_nindexes",
                "coll_storageSize",
                "validator_json",
                "timeseries_json",
                "view_on",
                "scan_error",
                "catalog_error",
            ]
        ]
    )

if not fields_df.empty:
    display(Markdown("## Observed Field Paths (first 200 rows)"))
    display(fields_df.head(200))

if not indexes_df.empty:
    display(Markdown("## Index Catalog"))
    display(indexes_df.head(200))

# Inventory Summary

,databases_visible,collections_and_views,indexes,observed_field_paths,generated_utc
0,4,8,20,186,2026-03-17T21:49:36.019134+00:00


## Databases

,database,sizeOnDisk,empty,db_collections,db_views,db_objects,db_dataSize,db_storageSize,db_indexes,db_indexSize,dbStats_error
0,admin,102400,False,2,0,3,6.280000e+02,40960.0,3,61440.0,None
1,config,110592,False,1,0,6,7.920000e+02,36864.0,2,73728.0,None
2,local,73728,False,1,0,4,9.497000e+03,36864.0,1,36864.0,None
3,quants_lab,1133674496,False,4,0,5704656,2.255310e+09,750882816.0,14,382791680.0,None


## Collections / Views

,database,collection,collection_type,estimated_document_count,scan_mode,documents_scanned_for_schema,schema_is_exhaustive_for_collection,observed_field_paths,coll_nindexes,coll_storageSize,validator_json,timeseries_json,view_on,scan_error,catalog_error
0,admin,system.users,collection,1,full,1,True,19.0,2,20480,null,null,None,NaN,None
1,admin,system.version,collection,2,full,2,True,3.0,1,20480,null,null,None,NaN,None
2,config,system.sessions,collection,6,full,0,False,NaN,2,36864,null,null,None,"OperationFailure: not authorized on config to execute command { find: ""system.sessions"", filter: {}, lsid: { id: UUID(""bf7818fd-a205-41a1-b521-153a72212700"") }, $db: ""config"" }, full error: {'ok':...",None
3,local,startup_log,collection,4,full,4,True,52.0,1,36864,null,null,None,NaN,None
4,quants_lab,candle_features,collection,32237,sample,5000,False,47.0,3,7954432,null,null,None,NaN,None
5,quants_lab,candles,collection,4841278,sample,5000,False,23.0,5,683122688,null,null,None,NaN,None
6,quants_lab,market_trades,collection,831132,sample,5000,False,23.0,4,59768832,null,null,None,NaN,None
7,quants_lab,symbol_metadata,collection,33,full,33,True,19.0,2,36864,null,null,None,NaN,None


## Observed Field Paths (first 200 rows)

,database,collection,path,parent_path,depth,documents_scanned_for_schema,documents_with_field,presence_ratio_in_scan,observed_required_in_scan,occurrences,non_null_occurrences,types,examples,scan_mode,estimated_document_count
0,admin,system.users,_id,NaN,0,1,1,1.000000,True,1,1,string,"""admin.admin""",full,1
1,admin,system.users,credentials,NaN,0,1,1,1.000000,True,1,1,object,"{""SCRAM-SHA-1"": {""iterationCount"": 10000, ""salt"": ""7NaliXi2Wn5Ivkgb/1XS5Q=="", ""storedKey"": ""s77puzNwvzhD8Zv2MM3JB93EVBI="", ""serverKey"": ""Gmh0J2vEpWpXxrjq9X55IPxtJGs=""}, ""SCRAM-SHA-256"": {""iteratio...",full,1
2,admin,system.users,credentials.SCRAM-SHA-1,credentials,1,1,1,1.000000,True,1,1,object,"{""iterationCount"": 10000, ""salt"": ""7NaliXi2Wn5Ivkgb/1XS5Q=="", ""storedKey"": ""s77puzNwvzhD8Zv2MM3JB93EVBI="", ""serverKey"": ""Gmh0J2vEpWpXxrjq9X55IPxtJGs=""}",full,1
3,admin,system.users,credentials.SCRAM-SHA-1.iterationCount,credentials.SCRAM-SHA-1,2,1,1,1.000000,True,1,1,int,10000,full,1
4,admin,system.users,credentials.SCRAM-SHA-1.salt,credentials.SCRAM-SHA-1,2,1,1,1.000000,True,1,1,string,"""7NaliXi2Wn5Ivkgb/1XS5Q==""",full,1
5,admin,system.users,credentials.SCRAM-SHA-1.serverKey,credentials.SCRAM-SHA-1,2,1,1,1.000000,True,1,1,string,"""Gmh0J2vEpWpXxrjq9X55IPxtJGs=""",full,1
6,admin,system.users,credentials.SCRAM-SHA-1.storedKey,credentials.SCRAM-SHA-1,2,1,1,1.000000,True,1,1,string,"""s77puzNwvzhD8Zv2MM3JB93EVBI=""",full,1
7,admin,system.users,credentials.SCRAM-SHA-256,credentials,1,1,1,1.000000,True,1,1,object,"{""iterationCount"": 15000, ""salt"": ""WmYSY1lZx+NZC0iwUKr+bGbEAKcx4bzLVtT1cQ=="", ""storedKey"": ""6z8tfXt/PBHz30PXC3a4hgVzeU2HYILlxM/llhF0JwI="", ""serverKey"": ""BFoRy94qcT5CYFdrIEsAi2TCPPUmhKj9RA37sLjLh9U=""}",full,1
8,admin,system.users,credentials.SCRAM-SHA-256.iterationCount,credentials.SCRAM-SHA-256,2,1,1,1.000000,True,1,1,int,15000,full,1
9,admin,system.users,credentials.SCRAM-SHA-256.salt,credentials.SCRAM-SHA-256,2,1,1,1.000000,True,1,1,string,"""WmYSY1lZx+NZC0iwUKr+bGbEAKcx4bzLVtT1cQ==""",full,1


## Index Catalog

,database,collection,index_name,index_keys,unique,sparse,hidden,expireAfterSeconds,partialFilterExpression_json,wildcardProjection_json,raw_index_json
0,admin,system.users,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"
1,admin,system.users,user_1_db_1,"user:1, db:1",True,False,False,NaN,null,null,"{""v"": 2, ""key"": {""user"": 1, ""db"": 1}, ""name"": ""user_1_db_1"", ""unique"": true}"
2,admin,system.version,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"
3,config,system.sessions,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"
4,config,system.sessions,lsidTTLIndex,lastUse:1,False,False,False,1800.0,null,null,"{""v"": 2, ""key"": {""lastUse"": 1}, ""name"": ""lsidTTLIndex"", ""expireAfterSeconds"": 1800}"
5,local,startup_log,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"
6,quants_lab,candle_features,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"
7,quants_lab,candle_features,idx_connector_pair_interval_ts,"connector:1, trading_pair:1, interval:1, timestamp:1",True,False,False,NaN,null,null,"{""v"": 2, ""key"": {""connector"": 1, ""trading_pair"": 1, ""interval"": 1, ""timestamp"": 1}, ""name"": ""idx_connector_pair_interval_ts"", ""unique"": true}"
8,quants_lab,candle_features,idx_latest_ts,"connector:1, trading_pair:1, interval:1, timestamp:-1",False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""connector"": 1, ""trading_pair"": 1, ""interval"": 1, ""timestamp"": -1}, ""name"": ""idx_latest_ts""}"
9,quants_lab,candles,_id_,_id:1,False,False,False,NaN,null,null,"{""v"": 2, ""key"": {""_id"": 1}, ""name"": ""_id_""}"


In [5]:
# Export tabular artifacts
databases_csv = OUTPUT_DIR / "databases.csv"
collections_csv = OUTPUT_DIR / "collections.csv"
indexes_csv = OUTPUT_DIR / "indexes.csv"
fields_csv = OUTPUT_DIR / "field_paths.csv"
summary_csv = OUTPUT_DIR / "summary.csv"
run_metadata_json = OUTPUT_DIR / "run_metadata.json"
report_md = OUTPUT_DIR / "schema_inventory_report.md"
readme_md = OUTPUT_DIR / "README.md"

databases_df.to_csv(databases_csv, index=False)
collections_df.to_csv(collections_csv, index=False)
indexes_df.to_csv(indexes_csv, index=False)
fields_df.to_csv(fields_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

run_metadata = {
    "generated_started_utc": run_started_utc,
    "generated_finished_utc": run_finished_utc,
    "mongo_uri_redacted": MONGO_URI.split("@")[-1] if "@" in MONGO_URI else MONGO_URI,
    "config": {
        "INCLUDE_DATABASES": INCLUDE_DATABASES,
        "EXCLUDE_DATABASES": EXCLUDE_DATABASES,
        "SKIP_SYSTEM_DATABASES": SKIP_SYSTEM_DATABASES,
        "SCAN_POLICY": SCAN_POLICY,
        "ADAPTIVE_FULL_SCAN_MAX_DOCS": ADAPTIVE_FULL_SCAN_MAX_DOCS,
        "SAMPLE_SCAN_SIZE": SAMPLE_SCAN_SIZE,
        "MAX_SAMPLE_DOCS_PER_COLLECTION": MAX_SAMPLE_DOCS_PER_COLLECTION,
        "SAMPLE_DOCUMENT_STRATEGY": SAMPLE_DOCUMENT_STRATEGY,
        "FIELD_EXAMPLE_LIMIT": FIELD_EXAMPLE_LIMIT,
        "TRUNCATE_LONG_VALUES_AT": TRUNCATE_LONG_VALUES_AT,
    },
    "counts": {
        "databases_visible": len(databases_df),
        "collections_and_views": int(collections_df["collection"].notna().sum()) if not collections_df.empty else 0,
        "indexes": int(indexes_df["index_name"].notna().sum()) if not indexes_df.empty else 0,
        "observed_field_paths": len(fields_df),
    },
}
save_json_file(run_metadata_json, run_metadata)

report_lines = []
report_lines.append("# MongoDB Estate Schema Inventory Report")
report_lines.append("")
report_lines.append(f"Generated UTC: {run_finished_utc}")
report_lines.append("")
report_lines.append("## Summary")
report_lines.append("")
for key, value in summary_df.iloc[0].to_dict().items():
    report_lines.append(f"- **{key}**: {value}")
report_lines.append("")
report_lines.append("## Notes")
report_lines.append("")
report_lines.append("- MongoDB is schemaless by default; field-path results are **observed**, not guaranteed unless validators are enforced.")
report_lines.append("- `schema_is_exhaustive_for_collection = True` only indicates the collection was fully scanned under the current settings.")
report_lines.append("- When a collection is sampled, the schema report may omit rare fields.")
report_lines.append("")
report_lines.append("## Databases")
report_lines.append("")
report_lines.append(
    dataframe_to_markdown(
        databases_df[
            [
                "database",
                "sizeOnDisk",
                "empty",
                "db_collections",
                "db_views",
                "db_objects",
                "db_dataSize",
                "db_storageSize",
                "db_indexes",
                "db_indexSize",
                "dbStats_error",
            ]
        ],
        max_rows=200,
    )
)
report_lines.append("")
report_lines.append("## Collections / Views")
report_lines.append("")
report_lines.append(
    dataframe_to_markdown(
        collections_df[
            [
                "database",
                "collection",
                "collection_type",
                "estimated_document_count",
                "scan_mode",
                "documents_scanned_for_schema",
                "schema_is_exhaustive_for_collection",
                "observed_field_paths",
                "coll_nindexes",
                "coll_storageSize",
                "validator_json",
                "timeseries_json",
                "view_on",
                "sample_file",
                "schema_file",
                "scan_error",
                "catalog_error",
            ]
        ] if not collections_df.empty else collections_df,
        max_rows=500,
    )
)
report_lines.append("")
report_lines.append("## Indexes")
report_lines.append("")
report_lines.append(
    dataframe_to_markdown(
        indexes_df[
            [
                "database",
                "collection",
                "index_name",
                "index_keys",
                "unique",
                "sparse",
                "hidden",
                "expireAfterSeconds",
                "partialFilterExpression_json",
            ]
        ] if not indexes_df.empty else indexes_df,
        max_rows=500,
    )
)
report_lines.append("")
report_lines.append("## Observed Field Paths")
report_lines.append("")
report_lines.append(
    dataframe_to_markdown(
        fields_df[
            [
                "database",
                "collection",
                "path",
                "parent_path",
                "depth",
                "documents_scanned_for_schema",
                "documents_with_field",
                "presence_ratio_in_scan",
                "observed_required_in_scan",
                "types",
                "examples",
                "scan_mode",
            ]
        ] if not fields_df.empty else fields_df,
        max_rows=1000,
    )
)

report_md.write_text("\n".join(report_lines), encoding="utf-8")

readme_lines = [
    "# Output Files",
    "",
    "- `summary.csv` — one-row run summary",
    "- `databases.csv` — database-level inventory",
    "- `collections.csv` — collection/view inventory and scan metadata",
    "- `indexes.csv` — index catalog",
    "- `field_paths.csv` — observed field paths and BSON types",
    "- `samples/<db>/<collection>.json` — sample documents for each collection",
    "- `schemas/<db>/<collection>.json` — observed schema payload for each collection",
    "- `schema_inventory_report.md` — markdown summary report",
    "- `run_metadata.json` — redacted runtime metadata and config",
]
readme_md.write_text("\n".join(readme_lines), encoding="utf-8")

display(Markdown("## Files Written"))
display(
    pd.DataFrame(
        [
            {"artifact": "summary.csv", "path": str(summary_csv)},
            {"artifact": "databases.csv", "path": str(databases_csv)},
            {"artifact": "collections.csv", "path": str(collections_csv)},
            {"artifact": "indexes.csv", "path": str(indexes_csv)},
            {"artifact": "field_paths.csv", "path": str(fields_csv)},
            {"artifact": "schema_inventory_report.md", "path": str(report_md)},
            {"artifact": "run_metadata.json", "path": str(run_metadata_json)},
            {"artifact": "README.md", "path": str(readme_md)},
        ]
    )
)

print(f"Artifacts written under: {OUTPUT_DIR.resolve()}")

## Files Written

,artifact,path
0,summary.csv,mongo_schema_inventory_output/summary.csv
1,databases.csv,mongo_schema_inventory_output/databases.csv
2,collections.csv,mongo_schema_inventory_output/collections.csv
3,indexes.csv,mongo_schema_inventory_output/indexes.csv
4,field_paths.csv,mongo_schema_inventory_output/field_paths.csv
5,schema_inventory_report.md,mongo_schema_inventory_output/schema_inventory_report.md
6,run_metadata.json,mongo_schema_inventory_output/run_metadata.json
7,README.md,mongo_schema_inventory_output/README.md


Artifacts written under: /quants-lab/research_notebooks/market_lab/pmm_dynamic/notebooks/mongo_schema_inventory_output


In [6]:
client.close()
print("MongoDB client closed.")

MongoDB client closed.
